# PRO3342 Modelagem e Simulação de Sistemas de Produção
## Projeto BVE - Modelo computacional: Navios

### Grupo G112

1. Francisco Rocha Freitas Brandão (18404341)
2. Inigo Anduaga (18404466)
3. Kepa Danis (18404719)
4. Vasco Oliveira (18440540)

Neste lab, o modelo conceitual desenvolvido no Lab 07 constitui a referência para a implementação do modelo computacional dos navios.

## 1. Objetivo

O modelo representa a operação dos navios no Porto Mar Azul e gera a demanda de combustível que alimentará, nas etapas seguintes, os modelos das barcaças e do PCO. Para cada terminal j (classe de navios j, com `c_j` berços equivalentes), simulam-se chegadas de Poisson, espera em fila FIFO no fundeio, manobra de entrada, operação, manobra de saída e saída do porto, ou seja, M sistemas `M/G/c_j` independentes. O abastecimento não altera a operação do navio; portanto, as barcaças e o PCO **não** fazem parte deste modelo.

**Entradas** (pasta `entrada/`): `terminais.csv`, `cenario.csv` e `distancias.csv`. Os dois primeiros são usados pelo modelo dos navios; `distancias.csv` fica preparado para o modelo das barcaças e não é lido aqui. `terminais.csv` e `distancias.csv` são gerados a partir do arquivo oficial `D08_2_BVE_MComp_Navios_-_Dados.md` (seção 3.1).

**Saída** (pasta `saida/`): `log_navios.csv`, uma linha por navio, conforme o contrato de saída do modelo conceitual (seção 5 do documento de referência do Lab 07).

### Relação com o modelo conceitual do Lab 07

O documento do grupo (`Modelo_Conceitual__g112`) descreve o sistema completo (navios, barcaças, base de carregamento e PCO) e não define os elementos específicos do processo dos navios (terminais, taxas de chegada, manobras, contrato do log). Esses elementos estão definidos no modelo conceitual de referência do processo de demanda (`D07_8_BVE_MConc_Navios`), que incorpora as correções do professor e é a especificação adotada nesta implementação. A tabela resume a comparação:

| Tema | Modelo do grupo (Lab 07) | Modelo de referência revisado | Tratamento em D08 |
|:--|:--|:--|:--|
| Navio retido até o fim do abastecimento (saída em `max(d2, fim do abastecimento)`) | Premissa do grupo | O abastecimento não altera a operação do navio; o navio sai ao fim da operação | Segue o modelo revisado: sem retenção |
| Atraso aleatório de atracação (δ) | Variável aleatória | Não existe; manobra de entrada com tempo constante por classe | Não implementado |
| Janela `[d1, d2]` | Atributos do pedido, com duração de estadia aleatória | Janela = período de operação no terminal (`t_ini_operacao`, `t_fim_operacao`) | Janela derivada da operação |
| Intervalo entre pedidos `I_C` e antecedência `A_C` aleatórios | Aleatórios | Chegadas de Poisson por terminal; antecedência constante por classe (48 h) | Exponencial e constante, lidas de `terminais.csv` |
| Pedido de combustível | Um por navio comprador | Sorteio de comprador (Bernoulli) e volume na notificação | Mantido o pedido único por navio; sorteado na notificação |
| Fundeio com capacidade ilimitada | Sim | Sim; fila FIFO por terminal, sem desistência | Mantido |
| Operação 24 h, sem restrição de tráfego ou clima | Premissa | Compatível | Mantido |
| Prioridade, data acordada, tipo passageiros | Atributos do pedido (PCO) | Fora do escopo do processo dos navios | Não implementados (dependem do PCO) |
| Horizonte | Não especificado | 500 dias no documento de referência | **365 dias**, conforme a instrução desta atividade (D08), parametrizado em `cenario.csv` |

### Pontos não definidos nos materiais e premissas adotadas

Os itens abaixo não estão definidos no modelo conceitual nem nos dados oficiais; foram assumidos e estão registrados em `cenario.csv` ou na implementação, para revisão do grupo:

1. `data_inicio` = 2026-01-01 00:00:00 (apenas referência de calendário para converter o relógio em data-hora).
2. `semente` = 20260 e `n_rep` = 1 (uma replicação do cenário-base).
3. O porto inicia vazio e não há período de aquecimento (nenhuma chegada nas primeiras 48 h, pois a antecedência da notificação é de 48 h).
4. O log contém os navios que concluem a manobra de saída até o fim do horizonte (conforme o pseudocódigo, o registro é gravado ao final do processo do navio). Os navios notificados e ainda em curso no horizonte são contabilizados em separado na seção 4.
5. `id_navio` tem formato `N000001`, sequencial por ordem de notificação, único dentro de cada cenário e replicação.
6. Taxas convertidas de navios/dia para navios/h e manobras convertidas de hh:mm para horas ao gerar `terminais.csv`.
7. Além das distribuições da convenção do modelo conceitual (uniforme, exponencial, triangular, erlang, lognormal), o código aceita `constante`, usada apenas nos testes com tempos determinísticos.

## 2. Prompt

**Ferramenta de IA:** Claude (Anthropic), via interface de chat. A IA foi utilizada para auditar os arquivos da atividade, comparar o modelo conceitual do grupo com o modelo de referência, escrever o código da preparação das entradas e do modelo SimPy, escrever e executar os testes de verificação e montar este notebook. As decisões de modelagem seguem os documentos listados na seção 6. **Revisão do grupo:** [a preencher pelo grupo: partes revisadas e forma de validação]. Os resultados apresentados foram obtidos por execução real do notebook.

O prompt original foi redigido em espanhol; abaixo está a sua versão em português, consolidada e organizada pelos requisitos.

```markdown
Atue como especialista em simulação de eventos discretos (SimPy). Implemente a atividade D08 - Projeto BVE: Modelo Computacional dos Navios (PRO3342).

OBJETIVO
Implementar o modelo computacional dos navios do Porto Mar Azul, que gera a demanda de combustível a ser usada depois pelos modelos das barcaças e do PCO. Não implementar barcaças, PCO nem abastecimento físico.

FONTES (ordem de prioridade)
1. Modelo conceitual revisado do Lab 07 (documento do grupo e modelo de referência com as correções do professor).
2. Feedback do professor. 3. Instruções de D08. 4. Dados oficiais de D08 (arquivo de dados em Markdown). 5. Materiais anteriores.
6. Suposições próprias só se imprescindíveis, sempre documentadas. Não substituir decisões do grupo por soluções genéricas. Se algo não estiver definido, indicar antes de assumir.

ENTRADAS (arquivos CSV, sem valores fixos no código)
- terminais.csv: id_terminal, id_classe, n_bercos, taxa_chegada (navios/h), antecedencia (h), tempo_manobra (h), dist_operacao e op_p1..p3 (h), prob_compra, dist_volume e vl_p1..p3 (t).
- cenario.csv: id_cenario, data_inicio, horizonte (dias), n_rep, semente.
- distancias.csv (formato longo, inclui a base): não usado neste modelo.
- Convenção de distribuições: uniforme(min, max), exponencial(média), triangular(min, moda, max), erlang(k, escala), lognormal(mu, sigma).
- Os parâmetros do cenário devem ser lidos dos arquivos de entrada: nada de número de berços, taxas, tempos, probabilidades, horizonte ou semente fixos no código. Gerar terminais.csv e distancias.csv a partir do arquivo oficial de dados.

ENTIDADES E RECURSOS
- Terminal: recurso com c_j berços equivalentes e uma fila FIFO por terminal, sem desistência e sem prioridade. Fundeio com capacidade ilimitada.
- Navio: entidade com id, classe e terminal. Pedido de combustível: atributo do navio (volume e janela), no máximo um por navio.

PROCESSO (um gerador independente por terminal)
1. Notificação: chegadas de Poisson (intervalos exponenciais com taxa do terminal). Cria o navio e, nesse instante, sorteia comprador (Bernoulli com prob_compra) e, se comprador, o volume da distribuição do terminal; caso contrário, volume zero.
2. Chegada ao fundeio após a antecedência (constante, 48 h no cenário-base).
3. Espera no fundeio por berço livre do terminal, em FIFO.
4. Manobra de entrada: o berço fica ocupado desde o início da manobra de entrada.
5. Operação no terminal, com tempo sorteado da distribuição do terminal. A janela de atendimento é o período de operação.
6. Manobra de saída, com tempo igual ao da entrada. O berço é liberado ao fim da manobra de saída.
7. Saída do porto e gravação do registro no log. O abastecimento não altera a operação do navio.
Variável de estado: N_j(t), navios da classe j no porto, da chegada ao fundeio até a saída.

SAÍDA: log_navios.csv, uma linha por navio, sem agregação, com as colunas
id_cenario, rep, id_navio, id_classe, id_terminal, t_notificacao, t_chegada, t_ini_manobra, t_ini_operacao, t_fim_operacao, t_fim_manobra (data-hora, a partir de data_inicio), comprador (0/1), volume_t (t, zero se comprador = 0).

HORIZONTE E SEMENTE: horizonte de 365 dias na replicação do cenário-base; semente fixa lida de cenario.csv, com resultados reproduzíveis.

ENTREGA: notebook com a estrutura do modelo do professor (Objetivo, Prompt, Modelo de simulação, Saída, Verificação, Referências), em português, executável do zero, com caminhos relativos e exportação para HTML.

VERIFICAÇÃO (semente fixa; para cada teste: objetivo, configuração, esperado, obtido, conclusão; executar de fato, sem inventar resultados)
Cobrir: geração de navios; relação notificação-chegada; capacidade dos berços; FIFO; manobra; operação; geração de compradores; geração de demanda; coerência temporal do log; contrato do log. Usar horizontes curtos, taxas reduzidas, tempos determinísticos ou terminal único quando necessário.

TRANSPARÊNCIA: não inventar nomes, dados, parâmetros, resultados ou decisões; indicar o uso de IA no relatório.
```

**Iterações subsequentes (resumo):** (1) auditoria dos seis arquivos e identificação do modelo conceitual do grupo e do modelo de referência do professor; (2) geração de `terminais.csv` e `distancias.csv` a partir do arquivo oficial de dados; (3) implementação do modelo SimPy e execução de teste inicial, após a qual o diagnóstico da simulação foi ampliado para incluir os navios notificados e ainda em curso no horizonte; (4) escrita e execução da bateria de verificação; (5) execução do notebook em kernel limpo e exportação para HTML.

## 3. Modelo de simulação

O modelo está em SimPy e é dividido em quatro blocos: (3.1) preparação dos arquivos de entrada a partir do arquivo oficial de dados; (3.2) leitura dos parâmetros; (3.3) simulação; (3.4) gravação do log. O fluxo é *arquivos de entrada → leitura dos parâmetros → modelo → simulação → log*: nenhum parâmetro do cenário aparece no código, e trocar de cenário exige apenas alterar os arquivos de `entrada/`.

### 3.1 Arquivos de entrada

`terminais.csv` e `distancias.csv` são derivados de `D08_2_BVE_MComp_Navios_-_Dados.md` por funções que interpretam o texto do arquivo; assim, os valores oficiais não são digitados no código. O arquivo `cenario.csv` contém o horizonte (365 dias), a data de referência, o número de replicações e a semente.

In [1]:
import re
from pathlib import Path

import pandas as pd

HORAS_POR_DIA = 24.0


def _num(txt):
    return float(txt.replace(".", "").replace(",", ".")) if re.search(r"\d\.\d{3}\b", txt) else float(txt.replace(",", "."))


def _secao(md, n):
    m = re.search(rf"^## {n}\.[^\n]*\n(.*?)(?=^## \d+\.|\Z)", md, flags=re.S | re.M)
    return m.group(1)


def _por_terminal(texto, padrao):
    out = {}
    for m in re.finditer(rf"^- (T\d+): {padrao}\s*$", texto, flags=re.M):
        out[m.group(1)] = m.groups()[1:]
    return out


def _horas(txt):
    h = re.search(r"(\d+)\s*h", txt)
    mi = re.search(r"(\d+)\s*min", txt)
    return (int(h.group(1)) if h else 0) + (int(mi.group(1)) if mi else 0) / 60.0


def ler_dados_md(caminho_md):
    md = Path(caminho_md).read_text(encoding="utf-8")

    s1 = _secao(md, 1)
    bercos = {k: int(v[0]) for k, v in _por_terminal(s1, r"(\d+) ber[çc]os?").items()}

    s3 = _secao(md, 3)
    taxa_dia = {k: _num(v[0]) for k, v in _por_terminal(s3, r"([\d,]+) navios?/dia").items()}

    s5 = _secao(md, 5)
    antecedencia = float(re.search(r"(\d+)\s*h", s5).group(1))

    s7 = _secao(md, 7)
    manobra = {k: _horas(v[0]) for k, v in _por_terminal(s7, r"(.+?)").items()}

    s8 = _secao(md, 8)
    operacao = {k: tuple(_num(x) for x in v) for k, v in
                _por_terminal(s8, r"Tri\(([\d,]+),\s*([\d,]+),\s*([\d,]+)\)\s*h").items()}

    s9 = _secao(md, 9)
    prob = {k: _num(v[0]) / 100.0 for k, v in _por_terminal(s9, r"(\d+)%").items()}

    s10 = _secao(md, 10)
    volume = {k: tuple(_num(x) for x in v) for k, v in
              _por_terminal(s10, r"Tri\(([\d,]+),\s*([\d,]+),\s*([\d,]+)\)\s*t").items()}

    linhas = [l for l in _secao(md, 11).splitlines() if l.startswith("|")]
    cab = [c.strip() for c in linhas[0].strip("|").split("|")]
    destinos = cab[1:]
    dist = []
    for l in linhas[2:]:
        cel = [c.strip() for c in l.strip("|").split("|")]
        for dest, v in zip(destinos, cel[1:]):
            dist.append({"origem": cel[0], "destino": dest, "distancia": _num(v)})

    return {"bercos": bercos, "taxa_dia": taxa_dia, "antecedencia": antecedencia, "manobra": manobra,
            "operacao": operacao, "prob": prob, "volume": volume, "distancias": pd.DataFrame(dist)}


def montar_terminais(d):
    linhas = []
    for t in sorted(d["bercos"], key=lambda x: int(x[1:])):
        linhas.append({
            "id_terminal": t,
            "id_classe": "C" + t[1:],
            "n_bercos": d["bercos"][t],
            "taxa_chegada": d["taxa_dia"][t] / HORAS_POR_DIA,
            "antecedencia": d["antecedencia"],
            "tempo_manobra": d["manobra"][t],
            "dist_operacao": "triangular",
            "op_p1": d["operacao"][t][0], "op_p2": d["operacao"][t][1], "op_p3": d["operacao"][t][2],
            "prob_compra": d["prob"][t],
            "dist_volume": "triangular",
            "vl_p1": d["volume"][t][0], "vl_p2": d["volume"][t][1], "vl_p3": d["volume"][t][2],
        })
    return pd.DataFrame(linhas)


def preparar_entradas(caminho_md, pasta_saida):
    pasta = Path(pasta_saida)
    pasta.mkdir(parents=True, exist_ok=True)
    d = ler_dados_md(caminho_md)
    terminais = montar_terminais(d)
    terminais.to_csv(pasta / "terminais.csv", index=False)
    d["distancias"].to_csv(pasta / "distancias.csv", index=False)
    return terminais, d["distancias"]


In [2]:
preparar_entradas("entrada/D08_2_BVE_MComp_Navios_-_Dados.md", "entrada")
display(pd.read_csv("entrada/terminais.csv"))
display(pd.read_csv("entrada/cenario.csv"))
display(pd.read_csv("entrada/distancias.csv").head(8))

,id_terminal,id_classe,n_bercos,taxa_chegada,antecedencia,tempo_manobra,dist_operacao,op_p1,op_p2,op_p3,prob_compra,dist_volume,vl_p1,vl_p2,vl_p3
0,T1,C1,3,0.104167,48.0,1.666667,triangular,12.0,15.0,19.0,0.70,triangular,600.0,1000.0,1800.0
1,T2,C2,5,0.166667,48.0,1.500000,triangular,14.0,18.0,22.0,0.60,triangular,800.0,1200.0,2000.0
2,T3,C3,4,0.187500,48.0,1.333333,triangular,9.0,12.0,15.0,0.50,triangular,400.0,700.0,1200.0
3,T4,C4,3,0.145833,48.0,1.333333,triangular,12.0,15.0,18.0,0.65,triangular,500.0,900.0,1600.0
4,T5,C5,2,0.083333,48.0,1.083333,triangular,9.0,12.0,15.0,0.55,triangular,300.0,600.0,1000.0
5,T6,C6,1,0.041667,48.0,0.916667,triangular,8.0,10.0,12.0,0.50,triangular,250.0,500.0,900.0


,id_cenario,data_inicio,horizonte,n_rep,semente
0,BASE,2026-01-01 00:00:00,365,1,20260


,origem,destino,distancia
0,Base,Base,0.0
1,Base,T1,2.1
2,Base,T2,3.7
3,Base,T3,5.8
4,Base,T4,7.5
5,Base,T5,9.2
6,Base,T6,10.6
7,T1,Base,2.1


`distancias.csv` (49 linhas, formato longo, inclui a base da BVE) não é utilizado pelo modelo dos navios; será usado no modelo das barcaças.

### 3.2 a 3.4 Modelo SimPy

Principais decisões de implementação:

- **Um gerador por terminal.** Cada terminal tem seu processo gerador (intervalos exponenciais com a taxa do terminal) e seu `simpy.Resource` com capacidade igual a `n_bercos`. O `Resource` do SimPy atende os pedidos em ordem de requisição, o que implementa a fila FIFO sem desistência e sem prioridade.
- **Notificação.** No instante da notificação são sorteados o indicador de comprador (Bernoulli) e o volume; o navio só chega ao fundeio após a antecedência constante do terminal.
- **Berço.** O berço é requisitado na chegada ao fundeio e só é liberado (saída do bloco `with`) ao final da manobra de saída; os instantes `t_ini_manobra`, `t_ini_operacao`, `t_fim_operacao` e `t_fim_manobra` são gravados nos pontos correspondentes.
- **Aleatoriedade reproduzível.** Para cada replicação, uma `SeedSequence([semente, rep])` gera fluxos independentes por terminal e por variável (chegadas, compra, volume, operação). Mudar o número de berços não altera as chegadas nem os volumes.
- **Estado e diagnóstico.** O modelo acompanha `N_j(t)`, a ocupação máxima de berços e a fila máxima por terminal, usados na verificação. O relógio interno é em horas e a conversão para data-hora ocorre ao montar o log.
- **Horizonte.** A simulação roda até `horizonte × 24 h`; só entram no log os navios que concluem a manobra de saída até esse instante.

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import simpy

HORAS_POR_DIA = 24.0

COLUNAS_LOG = ["id_cenario", "rep", "id_navio", "id_classe", "id_terminal",
               "t_notificacao", "t_chegada", "t_ini_manobra", "t_ini_operacao",
               "t_fim_operacao", "t_fim_manobra", "comprador", "volume_t"]
COLUNAS_TEMPO = [c for c in COLUNAS_LOG if c.startswith("t_")]


def carregar_parametros(pasta, id_cenario=None):
    pasta = Path(pasta)
    terminais = pd.read_csv(pasta / "terminais.csv")
    cenarios = pd.read_csv(pasta / "cenario.csv", parse_dates=["data_inicio"])
    if id_cenario is None:
        cenario = cenarios.iloc[0]
    else:
        cenario = cenarios.loc[cenarios["id_cenario"] == id_cenario].iloc[0]
    return terminais, cenario


def sortear(rng, dist, p1, p2=None, p3=None):
    if dist == "constante":
        return float(p1)
    if dist == "uniforme":
        return rng.uniform(p1, p2)
    if dist == "exponencial":
        return rng.exponential(p1)
    if dist == "triangular":
        return rng.triangular(p1, p2, p3)
    if dist == "erlang":
        return rng.gamma(p1, p2)
    if dist == "lognormal":
        return rng.lognormal(p1, p2)
    raise ValueError(f"Distribuição desconhecida: {dist}")


class Terminal:
    def __init__(self, env, linha, sequencias):
        self.id = linha["id_terminal"]
        self.classe = linha["id_classe"]
        self.n_bercos = int(linha["n_bercos"])
        self.taxa = float(linha["taxa_chegada"])
        self.antecedencia = float(linha["antecedencia"])
        self.manobra = float(linha["tempo_manobra"])
        self.dist_op = linha["dist_operacao"]
        self.par_op = (linha["op_p1"], linha["op_p2"], linha["op_p3"])
        self.prob_compra = float(linha["prob_compra"])
        self.dist_vol = linha["dist_volume"]
        self.par_vol = (linha["vl_p1"], linha["vl_p2"], linha["vl_p3"])
        self.bercos = simpy.Resource(env, capacity=self.n_bercos)
        self.rng_chegada, self.rng_compra, self.rng_volume, self.rng_operacao = (
            np.random.default_rng(s) for s in sequencias)
        self.N = 0
        self.n_notificados = 0
        self.ocupacao_max = 0
        self.fila_max = 0
        self.N_max = 0


def simular(terminais, cenario, rep):
    horizonte_h = float(cenario["horizonte"]) * HORAS_POR_DIA
    env = simpy.Environment()
    sementes = np.random.SeedSequence([int(cenario["semente"]), int(rep)]).spawn(4 * len(terminais))
    term = {}
    for i, (_, linha) in enumerate(terminais.iterrows()):
        term[linha["id_terminal"]] = Terminal(env, linha, sementes[4 * i:4 * i + 4])

    registros = []
    todos = []
    contador = {"navios": 0}

    def gerador_navios(t):
        while True:
            yield env.timeout(t.rng_chegada.exponential(1.0 / t.taxa))
            if env.now >= horizonte_h:
                break
            contador["navios"] += 1
            t.n_notificados += 1
            comprador = int(t.rng_compra.random() < t.prob_compra)
            volume = sortear(t.rng_volume, t.dist_vol, *t.par_vol) if comprador else 0.0
            reg = {"id_cenario": cenario["id_cenario"], "rep": int(rep),
                   "id_navio": f"N{contador['navios']:06d}", "id_classe": t.classe,
                   "id_terminal": t.id, "t_notificacao": env.now, "t_chegada": np.nan,
                   "t_ini_manobra": np.nan, "t_ini_operacao": np.nan, "t_fim_operacao": np.nan,
                   "t_fim_manobra": np.nan, "comprador": comprador, "volume_t": volume}
            todos.append(reg)
            env.process(navio(t, reg))

    def navio(t, reg):
        yield env.timeout(t.antecedencia)
        reg["t_chegada"] = env.now
        t.N += 1
        t.N_max = max(t.N_max, t.N)
        with t.bercos.request() as req:
            t.fila_max = max(t.fila_max, len(t.bercos.queue))
            yield req
            reg["t_ini_manobra"] = env.now
            t.ocupacao_max = max(t.ocupacao_max, t.bercos.count)
            yield env.timeout(t.manobra)
            reg["t_ini_operacao"] = env.now
            yield env.timeout(sortear(t.rng_operacao, t.dist_op, *t.par_op))
            reg["t_fim_operacao"] = env.now
            yield env.timeout(t.manobra)
            reg["t_fim_manobra"] = env.now
        t.N -= 1
        registros.append(reg)

    for t in term.values():
        env.process(gerador_navios(t))
    env.run(until=horizonte_h)

    diag = {
        "horizonte_h": horizonte_h,
        "n_notificados": {k: t.n_notificados for k, t in term.items()},
        "ocupacao_max": {k: t.ocupacao_max for k, t in term.items()},
        "fila_max": {k: t.fila_max for k, t in term.items()},
        "N_max": {k: t.N_max for k, t in term.items()},
        "n_registrados": len(registros),
        "n_total_notificados": contador["navios"],
        "notificados": montar_log(todos, cenario),
    }
    return montar_log(registros, cenario), diag


def montar_log(registros, cenario):
    log = pd.DataFrame(registros, columns=COLUNAS_LOG)
    origem = pd.Timestamp(cenario["data_inicio"])
    for c in COLUNAS_TEMPO:
        log[c] = origem + pd.to_timedelta(log[c], unit="h")
    log = log.sort_values("id_navio").reset_index(drop=True)
    log["rep"] = log["rep"].astype(int)
    log["comprador"] = log["comprador"].astype(int)
    log["volume_t"] = log["volume_t"].astype(float)
    return log


def gravar_log(log, caminho):
    Path(caminho).parent.mkdir(parents=True, exist_ok=True)
    log.to_csv(caminho, index=False, date_format="%Y-%m-%d %H:%M:%S.%f")


def executar(pasta_entrada, pasta_saida, id_cenario=None):
    terminais, cenario = carregar_parametros(pasta_entrada, id_cenario)
    logs, diags = [], []
    for rep in range(1, int(cenario["n_rep"]) + 1):
        log, diag = simular(terminais, cenario, rep)
        logs.append(log)
        diags.append(diag)
    log = pd.concat(logs, ignore_index=True)
    gravar_log(log, Path(pasta_saida) / "log_navios.csv")
    return log, diags


## 4. Saída

Execução da replicação do cenário-base (365 dias) a partir dos arquivos de entrada. O resultado é gravado em `saida/log_navios.csv`.

In [4]:
terminais, cenario = carregar_parametros("entrada")
print("Cenário:", cenario["id_cenario"], "| horizonte (dias):", cenario["horizonte"],
      "| replicações:", cenario["n_rep"], "| semente:", cenario["semente"])

log_navios, diagnosticos = executar("entrada", "saida")
diag_base = diagnosticos[0]
log_base = log_navios

print("Navios notificados no horizonte:", diag_base["n_total_notificados"])
print("Navios no log (saída concluída até o horizonte):", len(log_base))
print("Navios ainda em curso no horizonte (fora do log):", diag_base["n_total_notificados"] - len(log_base))
display(log_base.head(8))

Cenário: BASE | horizonte (dias): 365 | replicações: 1 | semente: 20260


Navios notificados no horizonte: 6467
Navios no log (saída concluída até o horizonte): 6407
Navios ainda em curso no horizonte (fora do log): 60


,id_cenario,rep,id_navio,id_classe,id_terminal,t_notificacao,t_chegada,t_ini_manobra,t_ini_operacao,t_fim_operacao,t_fim_manobra,comprador,volume_t
0,BASE,1,N000001,C2,T2,2026-01-01 00:43:57.791823859,2026-01-03 00:43:57.791823859,2026-01-03 00:43:57.791823859,2026-01-03 02:13:57.791823859,2026-01-03 16:34:25.207759959,2026-01-03 18:04:25.207759959,0,0.000000
1,BASE,1,N000002,C1,T1,2026-01-01 02:32:36.503655273,2026-01-03 02:32:36.503655273,2026-01-03 02:32:36.503655273,2026-01-03 04:12:36.503655274,2026-01-03 19:39:02.141721997,2026-01-03 21:19:02.141721994,1,1416.468533
2,BASE,1,N000003,C5,T5,2026-01-01 04:48:15.535278698,2026-01-03 04:48:15.535278698,2026-01-03 04:48:15.535278698,2026-01-03 05:53:15.535278697,2026-01-03 19:04:16.391714739,2026-01-03 20:09:16.391714738,0,0.000000
3,BASE,1,N000004,C4,T4,2026-01-01 07:20:52.931873248,2026-01-03 07:20:52.931873248,2026-01-03 07:20:52.931873248,2026-01-03 08:40:52.931873247,2026-01-03 23:07:18.563062317,2026-01-04 00:27:18.563062316,0,0.000000
4,BASE,1,N000005,C4,T4,2026-01-01 11:53:19.625942569,2026-01-03 11:53:19.625942569,2026-01-03 11:53:19.625942569,2026-01-03 13:13:19.625942571,2026-01-04 03:39:27.440770597,2026-01-04 04:59:27.440770599,1,823.652010
5,BASE,1,N000006,C3,T3,2026-01-01 13:27:40.901692982,2026-01-03 13:27:40.901692982,2026-01-03 13:27:40.901692982,2026-01-03 14:47:40.901692984,2026-01-04 04:32:16.680557209,2026-01-04 05:52:16.680557208,1,676.169772
6,BASE,1,N000007,C1,T1,2026-01-01 14:44:12.863689339,2026-01-03 14:44:12.863689339,2026-01-03 14:44:12.863689339,2026-01-03 16:24:12.863689340,2026-01-04 06:42:12.513803353,2026-01-04 08:22:12.513803354,1,1082.321575
7,BASE,1,N000008,C6,T6,2026-01-01 15:26:02.504099475,2026-01-03 15:26:02.504099475,2026-01-03 15:26:02.504099475,2026-01-03 16:21:02.504099473,2026-01-04 03:00:31.585920706,2026-01-04 03:55:31.585920704,0,0.000000


In [5]:
resumo = log_base.groupby("id_terminal").agg(
    navios=("id_navio", "count"),
    compradores=("comprador", "sum"),
    volume_demandado_t=("volume_t", "sum"),
    espera_media_h=("t_ini_manobra", lambda s: ((s - log_base.loc[s.index, "t_chegada"]).dt.total_seconds() / 3600).mean()),
    permanencia_media_h=("t_fim_manobra", lambda s: ((s - log_base.loc[s.index, "t_chegada"]).dt.total_seconds() / 3600).mean()),
).round(2)
display(resumo)
print(log_base.dtypes)

,navios,compradores,volume_demandado_t,espera_media_h,permanencia_media_h
id_terminal,,,,,
T1,881,603,689397.12,3.40,21.98
T2,1490,913,1213450.36,3.46,24.50
T3,1675,834,629848.04,2.85,17.52
T4,1285,878,878821.94,12.96,30.66
T5,696,372,237153.47,3.42,17.65
T6,380,181,98403.48,5.75,17.58


id_cenario                   str
rep                        int64
id_navio                     str
id_classe                    str
id_terminal                  str
t_notificacao     datetime64[ns]
t_chegada         datetime64[ns]
t_ini_manobra     datetime64[ns]
t_ini_operacao    datetime64[ns]
t_fim_operacao    datetime64[ns]
t_fim_manobra     datetime64[ns]
comprador                  int64
volume_t                 float64
dtype: object


**Campos do `log_navios.csv`** (uma linha por navio, registro bruto):

| Campo | Descrição |
|:--|:--|
| `id_cenario`, `rep` | Cenário e replicação |
| `id_navio` | Identificador único do navio (por ordem de notificação) |
| `id_classe`, `id_terminal` | Classe do navio e terminal de atracação |
| `t_notificacao` | Notificação da chegada; nesse instante são definidos `comprador` e `volume_t` |
| `t_chegada` | Chegada ao fundeio (notificação + antecedência) |
| `t_ini_manobra` | Início da manobra de entrada; o berço passa a estar ocupado |
| `t_ini_operacao` | Início da operação; abre a janela de atendimento |
| `t_fim_operacao` | Fim da operação; fecha a janela |
| `t_fim_manobra` | Fim da manobra de saída; libera o berço |
| `comprador` | 1 se o navio demanda combustível, 0 caso contrário |
| `volume_t` | Volume demandado em toneladas (zero se `comprador` = 0) |

Os instantes estão em data-hora, a partir de `data_inicio`. No modelo do PCO, `id_navio` e `id_classe` identificam o pedido, `id_terminal` é o local de atendimento, `t_notificacao` é a entrada do pedido na carteira, `t_ini_operacao` e `t_fim_operacao` definem a janela e `volume_t` a quantidade demandada. Indicadores como a espera no fundeio (`t_ini_manobra` − `t_chegada`) são calculados a partir do log.

## 5. Verificação do modelo

Todos os testes usam a semente fixa de `cenario.csv`. Os cenários de teste são obtidos modificando em memória as tabelas lidas dos arquivos de entrada (horizontes curtos, taxas ampliadas, tempos determinísticos ou terminal único), sem alterar a lógica do modelo. Os critérios estatísticos usam uma tolerância de 3 desvios-padrão. Cada teste apresenta objetivo, configuração, resultado esperado, resultado obtido e conclusão.

In [6]:
from IPython.display import Markdown

terminais_base, cenario_base = carregar_parametros("entrada")
origem = pd.Timestamp(cenario_base["data_inicio"])
RESULTADOS = []


def cen(**kw):
    c = cenario_base.copy()
    for k, v in kw.items():
        c[k] = v
    return c


def horas(serie):
    return serie.dt.total_seconds() / 3600


def relativo(serie):
    return horas(serie - origem)


def relatar(codigo, nome, objetivo, config, esperado, obtido, ok, conclusao):
    ok = bool(ok)
    RESULTADOS.append({"Teste": codigo, "Descrição": nome, "Resultado": "Aprovado" if ok else "Reprovado"})
    display(Markdown(
        f"**{codigo} – {nome}**\n\n"
        f"- **Objetivo:** {objetivo}\n"
        f"- **Configuração:** {config}\n"
        f"- **Esperado:** {esperado}\n"
        f"- **Obtido:** {obtido}\n"
        f"- **Conclusão:** {'Aprovado. ' if ok else 'REPROVADO. '}{conclusao}"))


def ocupacao_maxima(log, terminal):
    d = log[log["id_terminal"] == terminal]
    ev = [(t, 1) for t in d["t_ini_manobra"]] + [(t, -1) for t in d["t_fim_manobra"]]
    ev.sort(key=lambda x: (x[0], x[1]))
    atual = maximo = 0
    for _, delta in ev:
        atual += delta
        maximo = max(maximo, atual)
    return maximo


def esperas(log):
    return horas(log["t_ini_manobra"] - log["t_chegada"])

In [7]:
# V01 - Entradas fiéis ao arquivo oficial de dados
esperado_df = montar_terminais(ler_dados_md("entrada/D08_2_BVE_MComp_Navios_-_Dados.md"))
lido = pd.read_csv("entrada/terminais.csv")
try:
    pd.testing.assert_frame_equal(lido, esperado_df, check_exact=False, atol=1e-12)
    iguais = True
except AssertionError:
    iguais = False
dist_ok = len(pd.read_csv("entrada/distancias.csv")) == (len(terminais_base) + 1) ** 2
relatar("V01", "Entradas derivadas do arquivo oficial de dados",
        "Confirmar que `terminais.csv` e `distancias.csv` reproduzem os dados oficiais e que o modelo os lê do disco.",
        "`terminais.csv` relido e comparado com nova leitura do arquivo `.md`; contagem de linhas de `distancias.csv`; horizonte de `cenario.csv`.",
        "`terminais.csv` idêntico à nova leitura do `.md`; `distancias.csv` com (M+1)² linhas; horizonte de `cenario.csv` igual a 365 dias.",
        f"terminais idênticos: {iguais}; linhas em `distancias.csv`: {len(pd.read_csv('entrada/distancias.csv'))} (M = {len(terminais_base)}); horizonte: {cenario_base['horizonte']} dias.",
        iguais and dist_ok and float(cenario_base["horizonte"]) == 365,
        "Os parâmetros do cenário-base vêm do arquivo oficial e do `cenario.csv`.")

**V01 – Entradas derivadas do arquivo oficial de dados**

- **Objetivo:** Confirmar que `terminais.csv` e `distancias.csv` reproduzem os dados oficiais e que o modelo os lê do disco.
- **Configuração:** `terminais.csv` relido e comparado com nova leitura do arquivo `.md`; contagem de linhas de `distancias.csv`; horizonte de `cenario.csv`.
- **Esperado:** `terminais.csv` idêntico à nova leitura do `.md`; `distancias.csv` com (M+1)² linhas; horizonte de `cenario.csv` igual a 365 dias.
- **Obtido:** terminais idênticos: True; linhas em `distancias.csv`: 49 (M = 6); horizonte: 365 dias.
- **Conclusão:** Aprovado. Os parâmetros do cenário-base vêm do arquivo oficial e do `cenario.csv`.

In [8]:
# V02 - Geração de navios (Poisson por terminal)
H30 = 30
_, d30 = simular(terminais_base, cen(horizonte=H30), 1)
linhas, ok_cont = [], True
for _, r in terminais_base.iterrows():
    esp = r["taxa_chegada"] * H30 * 24
    n = d30["n_notificados"][r["id_terminal"]]
    ok_cont &= abs(n - esp) <= 3 * np.sqrt(esp)
    linhas.append(f"{r['id_terminal']}: {n} (esperado {esp:.1f} ± {3*np.sqrt(esp):.1f})")
ok_int, linhas_int = True, []
notif = diag_base["notificados"]
for _, r in terminais_base.iterrows():
    x = horas(notif.loc[notif["id_terminal"] == r["id_terminal"], "t_notificacao"].sort_values().diff().dropna())
    media_esp = 1 / r["taxa_chegada"]
    ok_int &= abs(x.mean() - media_esp) <= 3 * x.std() / np.sqrt(len(x)) and abs(x.std() / x.mean() - 1) <= 0.2
    linhas_int.append(f"{r['id_terminal']}: média {x.mean():.2f} h (esperado {media_esp:.2f}), CV {x.std()/x.mean():.2f}")
relatar("V02", "Geração de navios",
        "Verificar que cada terminal gera notificações como processo de Poisson com a taxa lida de `terminais.csv`.",
        f"(a) Contagem de notificações por terminal em {H30} dias; (b) intervalos entre notificações no cenário-base de 365 dias.",
        "(a) Contagem dentro de λ·T ± 3√(λ·T); (b) média dos intervalos igual a 1/λ (tolerância de 3 erros-padrão) e coeficiente de variação próximo de 1 (|CV − 1| ≤ 0,2), característico da exponencial.",
        "(a) " + "; ".join(linhas) + ". (b) " + "; ".join(linhas_int) + ".",
        ok_cont and ok_int,
        "O número e o espaçamento das notificações são compatíveis com um processo de Poisson de taxa λ em cada terminal.")

**V02 – Geração de navios**

- **Objetivo:** Verificar que cada terminal gera notificações como processo de Poisson com a taxa lida de `terminais.csv`.
- **Configuração:** (a) Contagem de notificações por terminal em 30 dias; (b) intervalos entre notificações no cenário-base de 365 dias.
- **Esperado:** (a) Contagem dentro de λ·T ± 3√(λ·T); (b) média dos intervalos igual a 1/λ (tolerância de 3 erros-padrão) e coeficiente de variação próximo de 1 (|CV − 1| ≤ 0,2), característico da exponencial.
- **Obtido:** (a) T1: 81 (esperado 75.0 ± 26.0); T2: 124 (esperado 120.0 ± 32.9); T3: 145 (esperado 135.0 ± 34.9); T4: 109 (esperado 105.0 ± 30.7); T5: 51 (esperado 60.0 ± 23.2); T6: 27 (esperado 30.0 ± 16.4). (b) T1: média 9.87 h (esperado 9.60), CV 1.01; T2: média 5.83 h (esperado 6.00), CV 0.98; T3: média 5.18 h (esperado 5.33), CV 1.00; T4: média 6.73 h (esperado 6.86), CV 1.03; T5: média 12.46 h (esperado 12.00), CV 1.02; T6: média 22.77 h (esperado 24.00), CV 0.96.
- **Conclusão:** Aprovado. O número e o espaçamento das notificações são compatíveis com um processo de Poisson de taxa λ em cada terminal.

In [9]:
# V03 - Notificação -> chegada
t_ant = terminais_base.copy()
t_ant["antecedencia"] = t_ant["antecedencia"] * 0.5
_, dA = simular(t_ant, cen(horizonte=H30), 1)
nt = dA["notificados"].merge(t_ant[["id_terminal", "antecedencia"]], on="id_terminal")
chegou = nt["t_chegada"].notna()
erro = (horas(nt.loc[chegou, "t_chegada"] - nt.loc[chegou, "t_notificacao"]) - nt.loc[chegou, "antecedencia"]).abs().max()
pend = nt.loc[~chegou]
pend_ok = ((relativo(pend["t_notificacao"]) + pend["antecedencia"]) >= H30 * 24 - 1e-9).all()
relatar("V03", "Relação notificação → chegada",
        "Confirmar que a chegada ao fundeio ocorre exatamente `antecedencia` horas após a notificação, com o valor lido do arquivo.",
        f"Antecedência de cada terminal reduzida à metade do valor do arquivo; horizonte de {H30} dias; todos os navios notificados.",
        "`t_chegada − t_notificacao` igual à antecedência da classe (erro máximo ≤ 1e-8 h); navios sem chegada registrada são apenas os notificados cuja chegada cairia após o horizonte.",
        f"Erro máximo: {erro:.2e} h em {int(chegou.sum())} navios; navios sem chegada no horizonte: {len(pend)} (todos com chegada prevista após o horizonte: {bool(pend_ok)}).",
        erro <= 1e-8 and pend_ok,
        "A antecedência é aplicada como constante lida do arquivo, e a mudança do arquivo alterou o comportamento sem alterar o código.")

**V03 – Relação notificação → chegada**

- **Objetivo:** Confirmar que a chegada ao fundeio ocorre exatamente `antecedencia` horas após a notificação, com o valor lido do arquivo.
- **Configuração:** Antecedência de cada terminal reduzida à metade do valor do arquivo; horizonte de 30 dias; todos os navios notificados.
- **Esperado:** `t_chegada − t_notificacao` igual à antecedência da classe (erro máximo ≤ 1e-8 h); navios sem chegada registrada são apenas os notificados cuja chegada cairia após o horizonte.
- **Obtido:** Erro máximo: 8.31e-13 h em 512 navios; navios sem chegada no horizonte: 25 (todos com chegada prevista após o horizonte: True).
- **Conclusão:** Aprovado. A antecedência é aplicada como constante lida do arquivo, e a mudança do arquivo alterou o comportamento sem alterar o código.

In [10]:
# V04 - Capacidade dos berços
t_sat = terminais_base.copy()
t_sat["taxa_chegada"] = t_sat["taxa_chegada"] * 3
log_sat, d_sat = simular(t_sat, cen(horizonte=H30), 1)
res = []
ok_cap = True
for _, r in terminais_base.iterrows():
    j, c = r["id_terminal"], int(r["n_bercos"])
    m_base, m_sat = ocupacao_maxima(log_base, j), ocupacao_maxima(log_sat, j)
    ok_cap &= (m_base <= c) and (m_sat == c) and d_sat["ocupacao_max"][j] == c and d_sat["fila_max"][j] > 0
    res.append(f"{j}: c = {c}, máx. no cenário-base = {m_base}, máx. sob carga triplicada = {m_sat} (fila máx. {d_sat['fila_max'][j]})")
relatar("V04", "Capacidade dos berços",
        "Verificar que nunca há mais navios ocupando berço do que `n_bercos` e que o limite é efetivamente atingido sob carga alta.",
        f"Ocupação calculada a partir do log (intervalo `t_ini_manobra` a `t_fim_manobra`) no cenário-base de 365 dias e em um cenário com taxas de chegada triplicadas ({H30} dias), em que há formação de fila.",
        "Ocupação máxima ≤ c_j no cenário-base; ocupação máxima = c_j sob carga alta, com fila não vazia; o monitor interno do modelo coincide com o log.",
        "; ".join(res) + ".",
        ok_cap,
        "A capacidade de cada terminal é respeitada e o limite é saturado quando a demanda excede os berços.")

**V04 – Capacidade dos berços**

- **Objetivo:** Verificar que nunca há mais navios ocupando berço do que `n_bercos` e que o limite é efetivamente atingido sob carga alta.
- **Configuração:** Ocupação calculada a partir do log (intervalo `t_ini_manobra` a `t_fim_manobra`) no cenário-base de 365 dias e em um cenário com taxas de chegada triplicadas (30 dias), em que há formação de fila.
- **Esperado:** Ocupação máxima ≤ c_j no cenário-base; ocupação máxima = c_j sob carga alta, com fila não vazia; o monitor interno do modelo coincide com o log.
- **Obtido:** T1: c = 3, máx. no cenário-base = 3, máx. sob carga triplicada = 3 (fila máx. 98); T2: c = 5, máx. no cenário-base = 5, máx. sob carga triplicada = 5 (fila máx. 171); T3: c = 4, máx. no cenário-base = 4, máx. sob carga triplicada = 4 (fila máx. 206); T4: c = 3, máx. no cenário-base = 3, máx. sob carga triplicada = 3 (fila máx. 187); T5: c = 2, máx. no cenário-base = 2, máx. sob carga triplicada = 2 (fila máx. 56); T6: c = 1, máx. no cenário-base = 1, máx. sob carga triplicada = 1 (fila máx. 21).
- **Conclusão:** Aprovado. A capacidade de cada terminal é respeitada e o limite é saturado quando a demanda excede os berços.

In [11]:
# V05 - FIFO
ok_fifo, res = True, []
for nome, lg in (("base", log_base), ("carga triplicada", log_sat)):
    for j, d in lg.groupby("id_terminal"):
        d = d.sort_values("t_chegada")
        mono = (d["t_ini_manobra"].diff().dropna() >= pd.Timedelta(0)).all()
        mesmo = (d["t_chegada"].rank(method="first").values == d["t_ini_manobra"].rank(method="first").values).all()
        ok_fifo &= bool(mono and mesmo)
    res.append(f"{nome}: {int((esperas(lg) > 1e-9).sum())} navios esperaram no fundeio")
relatar("V05", "Disciplina FIFO",
        "Confirmar que, em cada terminal, os navios iniciam a manobra de entrada na mesma ordem em que chegaram ao fundeio.",
        "Ordem de `t_chegada` comparada com a ordem de `t_ini_manobra` por terminal, no cenário-base e no cenário com carga triplicada (fila longa).",
        "Nenhuma ultrapassagem: `t_ini_manobra` não decrescente na ordem de chegada e postos idênticos, em todos os terminais.",
        "Nenhuma ultrapassagem encontrada; " + "; ".join(res) + ".",
        ok_fifo,
        "A fila de cada terminal é FIFO, sem prioridades nem desistências, inclusive em situação de espera efetiva.")

**V05 – Disciplina FIFO**

- **Objetivo:** Confirmar que, em cada terminal, os navios iniciam a manobra de entrada na mesma ordem em que chegaram ao fundeio.
- **Configuração:** Ordem de `t_chegada` comparada com a ordem de `t_ini_manobra` por terminal, no cenário-base e no cenário com carga triplicada (fila longa).
- **Esperado:** Nenhuma ultrapassagem: `t_ini_manobra` não decrescente na ordem de chegada e postos idênticos, em todos os terminais.
- **Obtido:** Nenhuma ultrapassagem encontrada; base: 3030 navios esperaram no fundeio; carga triplicada: 681 navios esperaram no fundeio.
- **Conclusão:** Aprovado. A fila de cada terminal é FIFO, sem prioridades nem desistências, inclusive em situação de espera efetiva.

In [12]:
# V06 - Manobra e ocupação do berço (caso determinístico, terminal único com 1 berço)
t1 = terminais_base[terminais_base["n_bercos"] == 1].copy()
t1["dist_operacao"] = "constante"
t1["op_p1"] = t1["op_p2"]
t1["taxa_chegada"] = t1["taxa_chegada"] * 3
log1, _ = simular(t1, cen(horizonte=20), 1)
m, op = float(t1["tempo_manobra"].iloc[0]), float(t1["op_p1"].iloc[0])
d = log1.sort_values("t_chegada")
cheg, ini = relativo(d["t_chegada"]).values, relativo(d["t_ini_manobra"]).values
esp_ini, esp_ini_errado, prev, prev_err = [], [], -np.inf, -np.inf
for c in cheg:
    i1 = max(c, prev); prev = i1 + m + op + m
    i2 = max(c, prev_err); prev_err = i2 + m + op
    esp_ini.append(i1); esp_ini_errado.append(i2)
erro_ini = np.abs(np.array(esp_ini) - ini).max()
dif_alt = int((np.abs(np.array(esp_ini_errado) - ini) > 1e-6).sum())
err_man_ent = (horas(log_base["t_ini_operacao"] - log_base["t_ini_manobra"]) - log_base["id_terminal"].map(terminais_base.set_index("id_terminal")["tempo_manobra"])).abs().max()
err_man_sai = (horas(log_base["t_fim_manobra"] - log_base["t_fim_operacao"]) - log_base["id_terminal"].map(terminais_base.set_index("id_terminal")["tempo_manobra"])).abs().max()
err_dur = (horas(d["t_fim_manobra"] - d["t_ini_manobra"]) - (2 * m + op)).abs().max()
relatar("V06", "Manobra de entrada/saída e ocupação do berço",
        "Verificar os tempos de manobra e que o berço fica ocupado da manobra de entrada ao fim da manobra de saída.",
        f"(a) Todos os navios do cenário-base: duração das manobras versus `tempo_manobra` do terminal. (b) Terminal único com 1 berço, operação constante ({op:g} h), taxa triplicada, 20 dias: início esperado calculado de forma independente pela recursão início_k = max(chegada_k, fim_(k−1)), com ocupação de 2·manobra + operação.",
        "(a) Manobras de entrada e saída iguais a `tempo_manobra` (erro ≤ 1e-8 h). (b) Início da manobra igual ao da recursão (erro ≤ 1e-6 h); duração no berço = 2·manobra + operação; a hipótese alternativa (berço liberado no fim da operação) deve produzir início diferente em vários navios.",
        f"(a) erro máximo na entrada {err_man_ent:.1e} h, na saída {err_man_sai:.1e} h ({len(log_base)} navios). (b) {len(d)} navios; erro máximo do início {erro_ini:.1e} h; erro da duração no berço {err_dur:.1e} h; a hipótese alternativa diferiria em {dif_alt} navios.",
        err_man_ent <= 1e-8 and err_man_sai <= 1e-8 and erro_ini <= 1e-6 and err_dur <= 1e-6 and dif_alt > 0,
        "O berço permanece ocupado durante as manobras de entrada e saída e a operação; os tempos de manobra seguem o arquivo de entrada.")

**V06 – Manobra de entrada/saída e ocupação do berço**

- **Objetivo:** Verificar os tempos de manobra e que o berço fica ocupado da manobra de entrada ao fim da manobra de saída.
- **Configuração:** (a) Todos os navios do cenário-base: duração das manobras versus `tempo_manobra` do terminal. (b) Terminal único com 1 berço, operação constante (10 h), taxa triplicada, 20 dias: início esperado calculado de forma independente pela recursão início_k = max(chegada_k, fim_(k−1)), com ocupação de 2·manobra + operação.
- **Esperado:** (a) Manobras de entrada e saída iguais a `tempo_manobra` (erro ≤ 1e-8 h). (b) Início da manobra igual ao da recursão (erro ≤ 1e-6 h); duração no berço = 2·manobra + operação; a hipótese alternativa (berço liberado no fim da operação) deve produzir início diferente em vários navios.
- **Obtido:** (a) erro máximo na entrada 8.3e-13 h, na saída 8.3e-13 h (6407 navios). (b) 36 navios; erro máximo do início 8.5e-13 h; erro da duração no berço 8.3e-13 h; a hipótese alternativa diferiria em 35 navios.
- **Conclusão:** Aprovado. O berço permanece ocupado durante as manobras de entrada e saída e a operação; os tempos de manobra seguem o arquivo de entrada.

In [13]:
# V07 - Tempo de operação
ok_op, res = True, []
for _, r in terminais_base.iterrows():
    d = log_base[log_base["id_terminal"] == r["id_terminal"]]
    x = horas(d["t_fim_operacao"] - d["t_ini_operacao"])
    a, b, c = r["op_p1"], r["op_p2"], r["op_p3"]
    media = (a + b + c) / 3
    ok_op &= x.min() >= a - 1e-9 and x.max() <= c + 1e-9 and abs(x.mean() - media) <= 3 * x.std() / np.sqrt(len(x))
    res.append(f"{r['id_terminal']}: [{x.min():.2f}; {x.max():.2f}] h, média {x.mean():.2f} (esperada {media:.2f})")
err_const = (horas(d1 := log1["t_fim_operacao"] - log1["t_ini_operacao"]) - op).abs().max()
relatar("V07", "Operação no terminal",
        "Verificar que a duração da operação segue a distribuição triangular do terminal e que `t_ini_operacao` abre e `t_fim_operacao` fecha a janela.",
        "Duração `t_fim_operacao − t_ini_operacao` do cenário-base por terminal (365 dias) e caso determinístico do teste V06.",
        "Valores entre mínimo e máximo; média igual a (mín + moda + máx)/3 com tolerância de 3 erros-padrão; com distribuição `constante`, duração igual ao parâmetro.",
        "; ".join(res) + f". Caso constante: erro máximo {err_const:.1e} h.",
        ok_op and err_const <= 1e-8,
        "As durações de operação respeitam a distribuição e os parâmetros lidos do arquivo.")

**V07 – Operação no terminal**

- **Objetivo:** Verificar que a duração da operação segue a distribuição triangular do terminal e que `t_ini_operacao` abre e `t_fim_operacao` fecha a janela.
- **Configuração:** Duração `t_fim_operacao − t_ini_operacao` do cenário-base por terminal (365 dias) e caso determinístico do teste V06.
- **Esperado:** Valores entre mínimo e máximo; média igual a (mín + moda + máx)/3 com tolerância de 3 erros-padrão; com distribuição `constante`, duração igual ao parâmetro.
- **Obtido:** T1: [12.15; 18.87] h, média 15.25 (esperada 15.33); T2: [14.13; 21.93] h, média 18.04 (esperada 18.00); T3: [9.07; 14.94] h, média 12.00 (esperada 12.00); T4: [12.10; 17.90] h, média 15.03 (esperada 15.00); T5: [9.15; 14.87] h, média 12.07 (esperada 12.00); T6: [8.19; 11.87] h, média 10.00 (esperada 10.00). Caso constante: erro máximo 0.0e+00 h.
- **Conclusão:** Aprovado. As durações de operação respeitam a distribuição e os parâmetros lidos do arquivo.

In [14]:
# V08 - Compradores
ok_cmp, res = True, []
for _, r in terminais_base.iterrows():
    d = log_base[log_base["id_terminal"] == r["id_terminal"]]
    p, n = r["prob_compra"], len(d)
    frac = d["comprador"].mean()
    ok_cmp &= abs(frac - p) <= 3 * np.sqrt(p * (1 - p) / n)
    res.append(f"{r['id_terminal']}: {frac:.3f} (p = {p:.2f}, n = {n})")
t_p0, t_p1 = terminais_base.copy(), terminais_base.copy()
t_p0["prob_compra"], t_p1["prob_compra"] = 0.0, 1.0
_, dp0 = simular(t_p0, cen(horizonte=H30), 1)
_, dp1 = simular(t_p1, cen(horizonte=H30), 1)
ext_ok = dp0["notificados"]["comprador"].sum() == 0 and dp1["notificados"]["comprador"].all()
relatar("V08", "Geração de compradores",
        "Verificar que o indicador de comprador é sorteado com a probabilidade `prob_compra` de cada terminal.",
        f"Fração de compradores por terminal no cenário-base (365 dias); casos extremos `prob_compra` = 0 e = 1 ({H30} dias).",
        "Fração observada dentro de p ± 3√(p(1−p)/n); sem compradores quando p = 0; todos compradores quando p = 1.",
        "; ".join(res) + f". Extremos: p = 0 → {int(dp0['notificados']['comprador'].sum())} compradores de {len(dp0['notificados'])}; p = 1 → {int(dp1['notificados']['comprador'].sum())} de {len(dp1['notificados'])}.",
        ok_cmp and ext_ok,
        "A condição de comprador segue a Bernoulli parametrizada por terminal.")

**V08 – Geração de compradores**

- **Objetivo:** Verificar que o indicador de comprador é sorteado com a probabilidade `prob_compra` de cada terminal.
- **Configuração:** Fração de compradores por terminal no cenário-base (365 dias); casos extremos `prob_compra` = 0 e = 1 (30 dias).
- **Esperado:** Fração observada dentro de p ± 3√(p(1−p)/n); sem compradores quando p = 0; todos compradores quando p = 1.
- **Obtido:** T1: 0.684 (p = 0.70, n = 881); T2: 0.613 (p = 0.60, n = 1490); T3: 0.498 (p = 0.50, n = 1675); T4: 0.683 (p = 0.65, n = 1285); T5: 0.534 (p = 0.55, n = 696); T6: 0.476 (p = 0.50, n = 380). Extremos: p = 0 → 0 compradores de 537; p = 1 → 537 de 537.
- **Conclusão:** Aprovado. A condição de comprador segue a Bernoulli parametrizada por terminal.

In [15]:
# V09 - Demanda de combustível
ok_vol, res = True, []
for _, r in terminais_base.iterrows():
    d = log_base[(log_base["id_terminal"] == r["id_terminal"]) & (log_base["comprador"] == 1)]
    x = d["volume_t"]
    a, b, c = r["vl_p1"], r["vl_p2"], r["vl_p3"]
    media = (a + b + c) / 3
    ok_vol &= x.min() >= a and x.max() <= c and abs(x.mean() - media) <= 3 * x.std() / np.sqrt(len(x))
    res.append(f"{r['id_terminal']}: [{x.min():.0f}; {x.max():.0f}] t, média {x.mean():.1f} (esperada {media:.1f})")
coer = ((log_base["comprador"] == 0) == (log_base["volume_t"] == 0)).all()
nn = diag_base["notificados"]
antes = nn[nn["t_chegada"].isna()]
define_not = len(antes) > 0 and antes["comprador"].isin([0, 1]).all() and ((antes["comprador"] == 1) == (antes["volume_t"] > 0)).all()
relatar("V09", "Geração da demanda (volume)",
        "Verificar o volume demandado (distribuição triangular por terminal) e que a demanda é definida na notificação.",
        "Volumes dos compradores do cenário-base por terminal (365 dias); coerência entre `comprador` e `volume_t`; navios notificados que ainda não chegaram ao fundeio no fim do horizonte.",
        "Volumes entre mínimo e máximo, média igual a (mín + moda + máx)/3 (3 erros-padrão); `volume_t` = 0 se e somente se `comprador` = 0; navios que apenas foram notificados já têm comprador e volume definidos.",
        "; ".join(res) + f". Coerência comprador/volume: {bool(coer)}. Navios só notificados no fim do horizonte: {len(antes)}, todos com demanda já definida: {bool(define_not)}.",
        ok_vol and coer and define_not,
        "A demanda de combustível é sorteada na notificação, com os parâmetros do arquivo, e registrada no log para uso posterior pelo PCO.")

**V09 – Geração da demanda (volume)**

- **Objetivo:** Verificar o volume demandado (distribuição triangular por terminal) e que a demanda é definida na notificação.
- **Configuração:** Volumes dos compradores do cenário-base por terminal (365 dias); coerência entre `comprador` e `volume_t`; navios notificados que ainda não chegaram ao fundeio no fim do horizonte.
- **Esperado:** Volumes entre mínimo e máximo, média igual a (mín + moda + máx)/3 (3 erros-padrão); `volume_t` = 0 se e somente se `comprador` = 0; navios que apenas foram notificados já têm comprador e volume definidos.
- **Obtido:** T1: [637; 1777] t, média 1143.3 (esperada 1133.3); T2: [841; 1969] t, média 1329.1 (esperada 1333.3); T3: [401; 1178] t, média 755.2 (esperada 766.7); T4: [505; 1579] t, média 1000.9 (esperada 1000.0); T5: [304; 991] t, média 637.5 (esperada 633.3); T6: [285; 875] t, média 543.7 (esperada 550.0). Coerência comprador/volume: True. Navios só notificados no fim do horizonte: 37, todos com demanda já definida: True.
- **Conclusão:** Aprovado. A demanda de combustível é sorteada na notificação, com os parâmetros do arquivo, e registrada no log para uso posterior pelo PCO.

In [16]:
# V10 - Coerência temporal do log
cols_t = ["t_notificacao", "t_chegada", "t_ini_manobra", "t_ini_operacao", "t_fim_operacao", "t_fim_manobra"]
ordem_ok = all((log_base[a] <= log_base[b]).all() for a, b in zip(cols_t[:-1], cols_t[1:]))
sem_nulos = not log_base.isna().any().any()
fim = origem + pd.Timedelta(days=float(cenario_base["horizonte"]))
dentro = (log_base["t_notificacao"] >= origem).all() and (log_base["t_notificacao"] < fim).all() and (log_base["t_fim_manobra"] <= fim).all()
ocup_ok = (horas(log_base["t_fim_manobra"] - log_base["t_ini_manobra"]) > 0).all()
relatar("V10", "Coerência temporal do log",
        "Verificar que os instantes de cada navio estão em ordem cronológica, completos e dentro do horizonte.",
        "Log do cenário-base de 365 dias.",
        "Para todos os navios: t_notificacao ≤ t_chegada ≤ t_ini_manobra ≤ t_ini_operacao ≤ t_fim_operacao ≤ t_fim_manobra; sem valores nulos; notificações dentro do horizonte e saídas até o fim do horizonte.",
        f"Ordem cronológica: {ordem_ok}; sem nulos: {sem_nulos}; dentro do horizonte: {bool(dentro)}; tempo no berço positivo: {bool(ocup_ok)} ({len(log_base)} navios).",
        ordem_ok and sem_nulos and dentro and ocup_ok,
        "Os registros temporais são consistentes com a sequência de eventos do modelo conceitual.")

**V10 – Coerência temporal do log**

- **Objetivo:** Verificar que os instantes de cada navio estão em ordem cronológica, completos e dentro do horizonte.
- **Configuração:** Log do cenário-base de 365 dias.
- **Esperado:** Para todos os navios: t_notificacao ≤ t_chegada ≤ t_ini_manobra ≤ t_ini_operacao ≤ t_fim_operacao ≤ t_fim_manobra; sem valores nulos; notificações dentro do horizonte e saídas até o fim do horizonte.
- **Obtido:** Ordem cronológica: True; sem nulos: True; dentro do horizonte: True; tempo no berço positivo: True (6407 navios).
- **Conclusão:** Aprovado. Os registros temporais são consistentes com a sequência de eventos do modelo conceitual.

In [17]:
# V11 - Contrato do log
contrato = ["id_cenario", "rep", "id_navio", "id_classe", "id_terminal", "t_notificacao", "t_chegada",
            "t_ini_manobra", "t_ini_operacao", "t_fim_operacao", "t_fim_manobra", "comprador", "volume_t"]
lido = pd.read_csv("saida/log_navios.csv", parse_dates=contrato[5:11])
cols_ok = list(log_base.columns) == contrato and list(lido.columns) == contrato
tipos_ok = (pd.api.types.is_integer_dtype(lido["rep"]) and pd.api.types.is_integer_dtype(lido["comprador"])
            and pd.api.types.is_float_dtype(lido["volume_t"])
            and all(pd.api.types.is_datetime64_any_dtype(lido[c]) for c in contrato[5:11]))
unico = log_base["id_navio"].is_unique and len(log_base) == diag_base["n_registrados"]
valores = set(log_base["comprador"].unique()) <= {0, 1} and (log_base["volume_t"] >= 0).all()
reler = len(lido) == len(log_base) and all((lido[c] - log_base[c]).abs().max() <= pd.Timedelta(microseconds=1) for c in contrato[5:11])
const = (lido["id_cenario"] == cenario_base["id_cenario"]).all() and (lido["rep"] == 1).all()
relatar("V11", "Contrato do `log_navios.csv`",
        "Confirmar que o arquivo gravado segue o contrato de saída do modelo conceitual.",
        "Arquivo `saida/log_navios.csv` do cenário-base, relido do disco.",
        "13 colunas, com os nomes e a ordem do contrato; instantes em data-hora; `comprador` em {0, 1}; `volume_t` ≥ 0; uma linha por navio com `id_navio` único; arquivo relido igual ao log em memória (tolerância de 1 µs).",
        f"Colunas conforme o contrato: {cols_ok}; tipos: {tipos_ok}; ids únicos e uma linha por navio: {unico}; valores válidos: {bool(valores)}; releitura fiel: {bool(reler)}; id_cenario/rep: {bool(const)}; {len(lido)} linhas.",
        cols_ok and tipos_ok and unico and valores and reler and const,
        "O log está em conformidade com o contrato e pode ser consumido pelas etapas seguintes.")

**V11 – Contrato do `log_navios.csv`**

- **Objetivo:** Confirmar que o arquivo gravado segue o contrato de saída do modelo conceitual.
- **Configuração:** Arquivo `saida/log_navios.csv` do cenário-base, relido do disco.
- **Esperado:** 13 colunas, com os nomes e a ordem do contrato; instantes em data-hora; `comprador` em {0, 1}; `volume_t` ≥ 0; uma linha por navio com `id_navio` único; arquivo relido igual ao log em memória (tolerância de 1 µs).
- **Obtido:** Colunas conforme o contrato: True; tipos: True; ids únicos e uma linha por navio: True; valores válidos: True; releitura fiel: True; id_cenario/rep: True; 6407 linhas.
- **Conclusão:** Aprovado. O log está em conformidade com o contrato e pode ser consumido pelas etapas seguintes.

In [18]:
# V12 - Reprodutibilidade
a1, _ = simular(terminais_base, cen(horizonte=H30), 1)
a2, _ = simular(terminais_base, cen(horizonte=H30), 1)
b, _ = simular(terminais_base, cen(horizonte=H30, semente=int(cenario_base["semente"]) + 1), 1)
c2, _ = simular(terminais_base, cen(horizonte=H30), 2)
relatar("V12", "Reprodutibilidade e semente",
        "Verificar que a mesma configuração reproduz o mesmo log e que a semente e a replicação alteram os resultados.",
        f"Quatro execuções de {H30} dias: duas iguais, uma com a semente do arquivo + 1, uma com a replicação 2.",
        "Logs idênticos para a mesma semente e replicação; logs diferentes para outra semente ou replicação.",
        f"Mesma configuração: idênticos = {a1.equals(a2)}; outra semente: diferentes = {not a1.drop(columns='rep').equals(b.drop(columns='rep'))}; outra replicação: diferentes = {not a1.drop(columns='rep').equals(c2.drop(columns='rep'))}.",
        a1.equals(a2) and not a1.drop(columns="rep").equals(b.drop(columns="rep")) and not a1.drop(columns="rep").equals(c2.drop(columns="rep")),
        "A semente fixa em `cenario.csv` torna os resultados reproduzíveis.")

**V12 – Reprodutibilidade e semente**

- **Objetivo:** Verificar que a mesma configuração reproduz o mesmo log e que a semente e a replicação alteram os resultados.
- **Configuração:** Quatro execuções de 30 dias: duas iguais, uma com a semente do arquivo + 1, uma com a replicação 2.
- **Esperado:** Logs idênticos para a mesma semente e replicação; logs diferentes para outra semente ou replicação.
- **Obtido:** Mesma configuração: idênticos = True; outra semente: diferentes = True; outra replicação: diferentes = True.
- **Conclusão:** Aprovado. A semente fixa em `cenario.csv` torna os resultados reproduzíveis.

In [19]:
# V13 - Parametrização
t_b2 = terminais_base.copy()
unico1 = t_b2["n_bercos"] == 1
t_b2.loc[unico1, "n_bercos"] = 2
t_b2.loc[unico1, "taxa_chegada"] = t_b2.loc[unico1, "taxa_chegada"] * 1.6
t_b1 = terminais_base.copy()
t_b1.loc[unico1, "taxa_chegada"] = t_b1.loc[unico1, "taxa_chegada"] * 1.6
l_b1, _ = simular(t_b1, cen(horizonte=120), 1)
l_b2, _ = simular(t_b2, cen(horizonte=120), 1)
j1 = terminais_base.loc[unico1, "id_terminal"].iloc[0]
e1 = esperas(l_b1[l_b1["id_terminal"] == j1]).mean(); e2 = esperas(l_b2[l_b2["id_terminal"] == j1]).mean()
n60, _ = simular(terminais_base, cen(horizonte=60), 1)
n120, _ = simular(terminais_base, cen(horizonte=120), 1)
cresc = len(n120) / len(n60)
def literais_numericos(codigo, acc):
    for k in codigo.co_consts:
        if hasattr(k, "co_consts"):
            literais_numericos(k, acc)
        elif isinstance(k, (int, float)) and not isinstance(k, bool):
            acc.add(float(k))
    return acc


literais = set()
for f in (carregar_parametros, Terminal.__init__, simular):
    literais_numericos(f.__code__, literais)
permitidos = {0.0, 1.0, 4.0}
relatar("V13", "Parametrização",
        "Verificar que o comportamento do modelo muda ao alterar os arquivos de entrada e que o código não contém valores do cenário.",
        f"(a) No terminal de 1 berço ({j1}), taxa × 1,6 com 1 berço e com 2 berços, mesma semente; (b) horizonte de 60 versus 120 dias; (c) análise dos literais numéricos das funções `carregar_parametros`, `Terminal` e `simular`.",
        "(a) Espera média menor com 2 berços; (b) número de navios aproximadamente proporcional ao horizonte (razão próxima de 2); (c) apenas literais estruturais (0, 1, 1.0 e 4 fluxos aleatórios por terminal).",
        f"(a) espera média {e1:.2f} h com 1 berço e {e2:.2f} h com 2 berços; (b) {len(n60)} e {len(n120)} navios (razão {cresc:.2f}); (c) literais encontrados: {sorted(literais)}.",
        e2 < e1 and 1.7 <= cresc <= 2.3 and literais <= permitidos,
        "Número de berços, taxas e horizonte são controlados pelos arquivos de entrada; o código só contém constantes estruturais.")

**V13 – Parametrização**

- **Objetivo:** Verificar que o comportamento do modelo muda ao alterar os arquivos de entrada e que o código não contém valores do cenário.
- **Configuração:** (a) No terminal de 1 berço (T6), taxa × 1,6 com 1 berço e com 2 berços, mesma semente; (b) horizonte de 60 versus 120 dias; (c) análise dos literais numéricos das funções `carregar_parametros`, `Terminal` e `simular`.
- **Esperado:** (a) Espera média menor com 2 berços; (b) número de navios aproximadamente proporcional ao horizonte (razão próxima de 2); (c) apenas literais estruturais (0, 1, 1.0 e 4 fluxos aleatórios por terminal).
- **Obtido:** (a) espera média 18.33 h com 1 berço e 0.99 h com 2 berços; (b) 962 e 2053 navios (razão 2.13); (c) literais encontrados: [0.0, 1.0, 4.0].
- **Conclusão:** Aprovado. Número de berços, taxas e horizonte são controlados pelos arquivos de entrada; o código só contém constantes estruturais.

In [20]:
# V14 - Coerência com a teoria de filas (utilização dos berços)
H = float(cenario_base["horizonte"]) * 24
ok_u, res = True, []
for _, r in terminais_base.iterrows():
    assert r["dist_operacao"] == "triangular"
    teo = r["taxa_chegada"] * ((r["op_p1"] + r["op_p2"] + r["op_p3"]) / 3 + 2 * r["tempo_manobra"]) / r["n_bercos"]
    d = log_base[log_base["id_terminal"] == r["id_terminal"]]
    obs = (horas(d["t_fim_manobra"] - d["t_ini_manobra"]).sum()) / (r["n_bercos"] * H)
    ok_u &= abs(obs - teo) <= 0.05
    res.append(f"{r['id_terminal']}: observada {obs:.3f}, teórica {teo:.3f}")
relatar("V14", "Utilização dos berços versus teoria",
        "Comparar a utilização observada com a esperada de um sistema M/G/c estável, ρ = λ·(E[operação] + 2·manobra)/c.",
        "Log do cenário-base de 365 dias; utilização = tempo total de ocupação dos berços ÷ (c × horizonte).",
        "Diferença absoluta ≤ 0,05 entre utilização observada e teórica em cada terminal (pequeno viés para baixo esperado pelo período inicial sem chegadas e pelos navios em curso no fim do horizonte).",
        "; ".join(res) + ".",
        ok_u,
        "A carga de trabalho simulada é coerente com a calculada analiticamente a partir dos parâmetros.")

**V14 – Utilização dos berços versus teoria**

- **Objetivo:** Comparar a utilização observada com a esperada de um sistema M/G/c estável, ρ = λ·(E[operação] + 2·manobra)/c.
- **Configuração:** Log do cenário-base de 365 dias; utilização = tempo total de ocupação dos berços ÷ (c × horizonte).
- **Esperado:** Diferença absoluta ≤ 0,05 entre utilização observada e teórica em cada terminal (pequeno viés para baixo esperado pelo período inicial sem chegadas e pelos navios em curso no fim do horizonte).
- **Obtido:** T1: observada 0.623, teórica 0.648; T2: observada 0.716, teórica 0.700; T3: observada 0.701, teórica 0.688; T4: observada 0.865, teórica 0.859; T5: observada 0.565, teórica 0.590; T6: observada 0.513, teórica 0.493.
- **Conclusão:** Aprovado. A carga de trabalho simulada é coerente com a calculada analiticamente a partir dos parâmetros.

In [21]:
resumo_testes = pd.DataFrame(RESULTADOS)
display(resumo_testes)
print("Testes aprovados:", int((resumo_testes["Resultado"] == "Aprovado").sum()), "de", len(resumo_testes))

,Teste,Descrição,Resultado
0,V01,Entradas derivadas do arquivo oficial de dados,Aprovado
1,V02,Geração de navios,Aprovado
2,V03,Relação notificação → chegada,Aprovado
3,V04,Capacidade dos berços,Aprovado
4,V05,Disciplina FIFO,Aprovado
5,V06,Manobra de entrada/saída e ocupação do berço,Aprovado
6,V07,Operação no terminal,Aprovado
7,V08,Geração de compradores,Aprovado
8,V09,Geração da demanda (volume),Aprovado
9,V10,Coerência temporal do log,Aprovado


Testes aprovados: 14 de 14


## 6. Referências

- PRO3342 Modelagem e Simulação de Sistemas de Produção. *D08 – Projeto BVE: Modelo Computacional dos Navios* (enunciado, notebook-modelo e arquivo de dados `D08_2_BVE_MComp_Navios_-_Dados.md`).
- Modelo conceitual de referência do processo de demanda (navios), `D07_8_BVE_MConc_Navios`, versão 1.0, 30/09/2026.
- Grupo G112. *Modelo Conceitual*, *Proposta de Consultoria* e *Solicitação de Dados* (Lab 07 e entregas anteriores).
- Costa, P. e Mesquita, M. (2023), citado na proposta do grupo, sobre bunkering no Porto de Santos.
- SimPy: documentação oficial (simpy.readthedocs.io), versão 4.
- Python, NumPy e pandas: documentação oficial.